## Part 1: Pipeline Setup & Functions
This section initializes the core ETL functions used in the production pipeline. It includes the `init_db` function to construct the database schema (Apps, Reviews, and Ingestion Batches), as well as the logic to extract 1,000 raw reviews per app, transform them by flagging low-quality data, and safely load them into the SQLite database.

In [7]:
import time
import sqlite3
import pandas as pd
import re
import datetime
from langdetect import detect, DetectorFactory
from google_play_scraper import Sort, reviews

DetectorFactory.seed = 0 

def is_english(text):
    try:
        text_for_detection = re.sub(r'[^\x00-\x7F]+', '', str(text))
        if len(text_for_detection.strip()) < 2:
            return False
        return detect(text_for_detection) == 'en'
    except:
        return False

def init_db(db_name='../data/play_store.db'):
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    cursor.executescript('''
        CREATE TABLE IF NOT EXISTS apps (
            app_id INTEGER PRIMARY KEY AUTOINCREMENT,
            app_name TEXT NOT NULL UNIQUE,
            platform TEXT DEFAULT 'Android',
            category TEXT,
            store_url TEXT
        );
        CREATE TABLE IF NOT EXISTS ingestion_batches (
            run_id INTEGER PRIMARY KEY AUTOINCREMENT, 
            batch_id TEXT,                            
            run_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            app_name TEXT, 
            total_scraped INTEGER,
            rows_inserted INTEGER,
            duplicates_skipped INTEGER,
            runtime_seconds REAL,
            status TEXT
        );
        CREATE TABLE IF NOT EXISTS reviews (
            internal_id INTEGER PRIMARY KEY AUTOINCREMENT,
            app_id INTEGER,
            run_id INTEGER, 
            store_review_id TEXT UNIQUE, 
            rating INTEGER,
            review_date TEXT,
            app_version TEXT,
            language_code TEXT,
            raw_text TEXT,
            clean_text TEXT,
            word_count INTEGER,
            thumbs_up_count INTEGER,
            is_low_signal BOOLEAN,
            is_duplicate BOOLEAN,
            is_english_flag BOOLEAN, 
            is_clean_baseline BOOLEAN,
            FOREIGN KEY(app_id) REFERENCES apps(app_id),
            FOREIGN KEY(run_id) REFERENCES ingestion_batches(run_id)
        );
    ''')
    conn.commit()
    conn.close()

def extract_android_reviews(app_name, app_id, target_reviews=1000):
    print(f"-> EXTRACT: Scraping {app_name} (Target: {target_reviews} reviews)...")
    app_reviews = []
    seen_ids = set()
    continuation_token = None
    
    while len(app_reviews) < target_reviews:
        result, continuation_token = reviews(
            app_id, lang='en', country='us', sort=Sort.NEWEST,
            count=1000, continuation_token=continuation_token
        )
        if not result: break
        for review in result:
            r_id = review['reviewId']
            if r_id not in seen_ids:
                seen_ids.add(r_id)
                app_reviews.append({
                    'store_review_id': r_id, 'rating': review['score'],
                    'review_date': review['at'], 'app_version': review.get('reviewCreatedVersion', 'Unknown'),
                    'language_code': 'en', 'raw_text': review['content'], 'thumbs_up_count': review['thumbsUpCount']
                })
        if len(app_reviews) >= target_reviews:
            app_reviews = app_reviews[:target_reviews]
            break
        
        # Respect the API limits during large pulls
        time.sleep(0.3)
            
    print(f"   Successfully extracted {len(app_reviews)} rows.")
    return pd.DataFrame(app_reviews)

def transform_data(df):
    print("-> TRANSFORM: Applying data quality flags...")
    if df.empty: return df
    df['review_date'] = df['review_date'].astype(str)
    df['clean_text'] = df['raw_text'].fillna("").astype(str).str.lower()
    df['word_count'] = df['clean_text'].str.split().str.len()
    df['is_duplicate'] = df.duplicated(subset=['store_review_id'], keep=False)
    df['is_low_signal'] = df['word_count'] < 3 
    df['is_english_flag'] = df['clean_text'].apply(is_english)
    df['is_clean_baseline'] = ~(df['is_duplicate'] | df['is_low_signal'] | ~df['is_english_flag'])
    return df

def load_to_sqlite(df, app_name, category, store_url, runtime_seconds, batch_id, db_name='../data/play_store.db'):
    print("-> LOAD: Connecting to database...")
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    
    cursor.execute("INSERT INTO ingestion_batches (batch_id, app_name, status) VALUES (?, ?, 'RUNNING')", (batch_id, app_name))
    current_run_id = cursor.lastrowid 
    
    cursor.execute('''INSERT OR IGNORE INTO apps (app_name, platform, category, store_url) VALUES (?, 'Android', ?, ?)''', (app_name, category, store_url))
    cursor.execute("SELECT app_id FROM apps WHERE app_name = ?", (app_name,))
    app_id = cursor.fetchone()[0]
    
    total_scraped = len(df)
    rows_inserted = 0
    if not df.empty:
        for index, row in df.iterrows():
            cursor.execute('''
                INSERT OR IGNORE INTO reviews (
                    app_id, run_id, store_review_id, rating, review_date, app_version, language_code, 
                    raw_text, clean_text, word_count, thumbs_up_count, is_low_signal, is_duplicate, 
                    is_english_flag, is_clean_baseline
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (
                app_id, current_run_id, row['store_review_id'], row['rating'], row['review_date'], 
                row['app_version'], row['language_code'], row['raw_text'], row['clean_text'], 
                row['word_count'], row['thumbs_up_count'], row['is_low_signal'], row['is_duplicate'], 
                row['is_english_flag'], row['is_clean_baseline']
            ))
            rows_inserted += cursor.rowcount
            
    duplicates_skipped = total_scraped - rows_inserted
    cursor.execute('''
        UPDATE ingestion_batches 
        SET total_scraped = ?, rows_inserted = ?, duplicates_skipped = ?, runtime_seconds = ?, status = 'SUCCESS' 
        WHERE run_id = ?
    ''', (total_scraped, rows_inserted, duplicates_skipped, runtime_seconds, current_run_id))
    conn.commit()
    conn.close()
    print(f"   Success! {rows_inserted} inserted. {duplicates_skipped} duplicates skipped.")

## Part 2: Portfolio Execution
We are executing a live ingestion run across our 5-app sample portfolio. The scraper will extract 1,000 reviews for each app (totaling 5,000 records). 

Because the pipeline is built to be idempotent, it will safely ignore any reviews it has already stored in previous batches, preventing data duplication while appending only fresh user feedback.

In [10]:
print("=== INITIALIZING DATABASE ===")
init_db()

APP_PORTFOLIO = {
    'WhatsApp':    {'id': 'com.whatsapp', 'category': 'Social & Communication'},
    'Gmail':       {'id': 'com.google.android.gm', 'category': 'Workspace & Productivity'},
    'Amazon':      {'id': 'com.amazon.mShop.android.shopping', 'category': 'E-Commerce & Marketplaces'},
    'PayPal':      {'id': 'com.paypal.android.p2pmobile', 'category': 'FinTech & Payments'},
    'Uber':        {'id': 'com.ubercab', 'category': 'Mobility, Travel & Maps'}
}

print("\n=== STARTING FULL PORTFOLIO PIPELINE RUN ===")
master_batch_id = datetime.datetime.now().strftime("BATCH_%Y%m%d_%H%M%S")

for app_name, app_info in APP_PORTFOLIO.items():
    print(f"--- Processing {app_name} ---")
    start_time = time.time() 
    target_url = f"https://play.google.com/store/apps/details?id={app_info['id']}"
    
    try:
        raw_data = extract_android_reviews(app_name, app_info['id'], target_reviews=1000)
        clean_data = transform_data(raw_data)
        
        runtime = round(time.time() - start_time, 2)
        
        load_to_sqlite(
            df=clean_data, 
            app_name=app_name, 
            category=app_info['category'], 
            store_url=target_url,
            runtime_seconds=runtime,
            batch_id=master_batch_id 
        )
        print("\n") 
        
    except Exception as e:
        print(f"FAILED to process {app_name}. Error: {e}\n")

print("=== FULL PORTFOLIO RUN COMPLETE ===")

=== INITIALIZING DATABASE ===

=== STARTING FULL PORTFOLIO PIPELINE RUN ===
--- Processing WhatsApp ---
-> EXTRACT: Scraping WhatsApp (Target: 1000 reviews)...
   Successfully extracted 1000 rows.
-> TRANSFORM: Applying data quality flags...
-> LOAD: Connecting to database...
   Success! 0 inserted. 1000 duplicates skipped.


--- Processing Gmail ---
-> EXTRACT: Scraping Gmail (Target: 1000 reviews)...
   Successfully extracted 1000 rows.
-> TRANSFORM: Applying data quality flags...
-> LOAD: Connecting to database...
   Success! 0 inserted. 1000 duplicates skipped.


--- Processing Amazon ---
-> EXTRACT: Scraping Amazon (Target: 1000 reviews)...
   Successfully extracted 1000 rows.
-> TRANSFORM: Applying data quality flags...
-> LOAD: Connecting to database...
   Success! 0 inserted. 1000 duplicates skipped.


--- Processing PayPal ---
-> EXTRACT: Scraping PayPal (Target: 1000 reviews)...
   Successfully extracted 1000 rows.
-> TRANSFORM: Applying data quality flags...
-> LOAD: Connect

## Part 3: Verification & Telemetry Export
With the tests complete, we query the `ingestion_batches` table to review the telemetry report. We expect to see two distinct batches, with the second batch explicitly showing a high `duplicates_skipped` count.

Finally, we export the resulting DataFrame as a `.png` file to our `assets/` folder for use in the project's README documentation.

In [12]:
import dataframe_image as dfi 

db_path = '../data/play_store.db'

with sqlite3.connect(db_path) as conn:
    # 1. Pull the Manager's Telemetry Report
    telemetry_query = """
        SELECT run_id, batch_id, app_name, run_date, total_scraped, rows_inserted, duplicates_skipped, runtime_seconds, status
        FROM ingestion_batches
        ORDER BY run_id DESC
    """
    telemetry_df = pd.read_sql(telemetry_query, conn)
    
    # 2. Pull a sample of clean, baseline data
    sample_reviews_query = """
        SELECT r.app_id, a.app_name, r.rating, r.clean_text, r.is_english_flag, r.run_id, b.batch_id
        FROM reviews r
        JOIN apps a ON r.app_id = a.app_id
        JOIN ingestion_batches b ON r.run_id = b.run_id
        WHERE r.is_clean_baseline = 1
        LIMIT 5
    """
    reviews_df = pd.read_sql(sample_reviews_query, conn)

# Display the results
print("=== TELEMETRY SUMMARY (TESTING BATCHES) ===")
display(telemetry_df) 

print("\n=== SAMPLE OF CLEAN BASELINE REVIEWS ===")
display(reviews_df)

=== TELEMETRY SUMMARY (TESTING BATCHES) ===


,run_id,batch_id,app_name,run_date,total_scraped,rows_inserted,duplicates_skipped,runtime_seconds,status
0,10,BATCH_20260805_014721,Uber,2026-08-05 08:47:38,1000,0,1000,3.81,SUCCESS
1,9,BATCH_20260805_014721,PayPal,2026-08-05 08:47:35,1000,0,1000,3.14,SUCCESS
2,8,BATCH_20260805_014721,Amazon,2026-08-05 08:47:31,1000,0,1000,2.78,SUCCESS
3,7,BATCH_20260805_014721,Gmail,2026-08-05 08:47:29,1000,0,1000,3.51,SUCCESS
4,6,BATCH_20260805_014721,WhatsApp,2026-08-05 08:47:25,1000,0,1000,4.20,SUCCESS
5,5,BATCH_20260805_014654,Uber,2026-08-05 08:47:12,1000,1000,0,3.82,SUCCESS
6,4,BATCH_20260805_014654,PayPal,2026-08-05 08:47:08,1000,1000,0,3.10,SUCCESS
7,3,BATCH_20260805_014654,Amazon,2026-08-05 08:47:05,1000,1000,0,2.84,SUCCESS
8,2,BATCH_20260805_014654,Gmail,2026-08-05 08:47:02,1000,1000,0,3.65,SUCCESS
9,1,BATCH_20260805_014654,WhatsApp,2026-08-05 08:46:58,1000,1000,0,3.71,SUCCESS



=== SAMPLE OF CLEAN BASELINE REVIEWS ===


,app_id,app_name,rating,clean_text,is_english_flag,run_id,batch_id
0,1,WhatsApp,5,please help me,1,1,BATCH_20260805_014654
1,1,WhatsApp,5,hi whatsapp give a update in that we will add ...,1,1,BATCH_20260805_014654
2,1,WhatsApp,4,whatsapp verification code not recieved proble...,1,1,BATCH_20260805_014654
3,1,WhatsApp,5,"whatsapp account in review,please give me help...",1,1,BATCH_20260805_014654
4,1,WhatsApp,1,hello whatsap team. my whatsap account got dis...,1,1,BATCH_20260805_014654
